# Project Report

# Introduction and description of the dataset


This project tackles a niche but visually interesting problem: distinguishing between
two types of Japanese plush toys based on the *Touhou Project* franchise —
**Fumo** and **Chocopuni**. Both represent the same characters, but differ
significantly in style, shape, and manufacturer.

| | Fumo (FumoFumo) | Chocopuni (Chokopuni) |
|---|---|---|
| **Manufacturer** | Gift (official) | Good Smile Company |
| **Style** | Classic, slightly angular, "mofumofu" | Round, chibi, soft "puni" shape |
| **Size** | ~20–25 cm (seated) | Smaller, more compact |
| **Origin** | Touhou Project (general) | Touhou LostWord (game) |
| **Face** | Flat face, embroidered eyes | Softer features, printed eyes |

Despite belonging to the same franchise, the two types have distinct silhouettes,
proportions, and textures — making this a realistic fine-grained visual classification task.


## Dataset

The Fumo and Chocopuni datasets consist of approximately 90% pictures taken by myself and friends.
The remaining images were scraped from the internet and used primarily as test data —
the goal being to verify that the model can recognize plushies it has never seen before,
not just the ones it was trained on. This gives a much more honest measure of real-world performance.

📁 **Dataset:** [Link to dataset](https://drive.google.com/drive/folders/1gp_d7FjI8-9Uuu579UwrCWxPdw3DeMX8)




### Dataset Collection

With only 4 chocopunis and 11 fumos available, reaching a meaningful dataset size
without causing overfitting was a real constraint.

Collection happened in three stages:

1. **Controlled environment** — black background, white light, rotating the plushie to capture all angles
2. **Second controlled environment** — white background, warm light, still clean studio-style shots
3. **Random environments** — various rooms, lighting conditions, backgrounds, and positions to simulate real-world photos

This brought the count to roughly ~60 images per class. To go further, I reached out to
friends and acquaintances who also own these plushies, which added more unique subjects
and backgrounds to the dataset.

### Final split

| Subset | Images | Purpose |
|---|---|---|
| Train | ~70% | Model training with augmentation |
| Validation | ~15% | Early stopping and hyperparameter decisions |
| Test | ~15% | Final evaluation only — never seen during training |



## Model Performance Summary

| | Model 1 — Custom CNN | Model 2 — EfficientNet-B0 | Model 3 — Fine-Tuned |
|---|---|---|---|
| **Test Accuracy** | 61.0% | 85.4% | 91.7% |
| **Loss** | 0.671 | 0.474 | 0.392 |
| **Macro F1** | 0.60 | 0.85 | 0.92 |
| **Chocopuni F1** | 0.65 | 0.85 | 0.92 |
| **Fumo F1** | 0.56 | 0.86 | 0.91 |
| **Bias** | Strong toward Chocopuni | Balanced | Balanced |

## Model 1 — Custom CNN

The baseline model was a small 3-block convolutional network trained entirely from
scratch. Test accuracy reached **61%** — only slightly above random chance (50%),
confirming the expected outcome for a network with no pretrained knowledge and a
small dataset.

The classification report revealed a clear bias: Chocopuni recall was 0.79 while
Fumo recall was only 0.45 — meaning the model missed more than half of all Fumo
images. The model appeared to latch onto superficial features (possibly background
color or overall brightness) rather than learning the actual shape differences
between the two classes.

The confusion matrix confirmed this: a disproportionate number of Fumo images were
predicted as Chocopuni, suggesting the decision boundary was skewed rather than
learned properly.

---

## Model 2 — EfficientNet-B0 as Feature Extractor

Switching to a pretrained EfficientNet-B0 backbone produced a dramatic improvement.
Test accuracy jumped to **85.4%** — a gain of **+24.4 percentage points** over Model 1.

Key differences from Model 1:
- The frozen EfficientNet layers provided rich, generalizable feature representations
  (edges, textures, shapes) learned from 1.2 million ImageNet images
- Only the last convolutional block and the classifier head were trained on our data,
  drastically reducing the risk of overfitting
- Stratified K-Fold cross-validation (3 folds) replaced the fixed train/val split,
  giving a more reliable estimate of generalization on a small dataset

The class bias present in Model 1 largely disappeared: both classes reached F1
scores of 0.85–0.86, and the gap between Chocopuni recall (0.89) and Fumo recall
(0.82) was small and acceptable.


## Model 3 — Fine-Tuned EfficientNet-B0

Fine-tuning the upper three convolutional blocks (features[6], [7], [8]) pushed
accuracy further to **91.7%** — a gain of **+6.3 percentage points** over Model 2.

Both classes are now nearly perfectly balanced (F1: 0.92 and 0.91), and the loss
dropped to 0.392, indicating higher prediction confidence. The diminishing returns
between Model 2 and 3 suggest the dataset is approaching its practical performance
ceiling — further gains would likely require more training data.

---


## Hyperparameter Exploration

Models 2 and 3 were tested under several different configurations during development:

| Parameter | Values tested | Final choice |
|---|---|---|
| `BATCH_SIZE` | 32, 64 | 32 (Model 3), 64 (Model 2) |
| `EPOCHS` | 20, 30, 40 | 20 (Model 2), 40 (Model 3) |
| `NUM_WORKERS` | 4, 10 | 4 (stable on Windows) |
| `N_FOLDS` | 3, 5 | 3 (Model 2), 5 (Model 3) |

Key observations from this exploration:
- **Batch size 64** with 3 folds worked well for Model 2 where only the head is trained —
  larger batches give more stable gradient estimates when fewer parameters are updated
- **Batch size 32** with 5 folds was preferred for Model 3 — smaller batches and more
  folds provide better regularization when more layers are unfrozen
- **NUM_WORKERS > 4** consistently caused `WinError 1455` on Windows due to each
  worker spawning a full PyTorch process; 4 workers was the reliable maximum
- **More epochs** (40 vs 20) were needed for Model 3 because fine-tuning pretrained
  layers with a very low learning rate (`1e-6`) converges significantly slower

---


## Challenges

### Switching to PyTorch

The project started with Keras (PyTorch backend) but encountered kernel crashes
related to the `Permute` layer and multiprocessing conflicts on Windows. From
Model 2 onwards the project moved to pure PyTorch with a manual training loop.
This required learning gradient handling, DataLoader mechanics, and writing
evaluation loops from scratch — but resulted in a more transparent and
debuggable setup.

### Dataset collection

The physical collection was limited by the number of plushies available. Shooting
in multiple environments (controlled studio vs. random real-world) was intentional
but introduced a train/test distribution mismatch: training images were mostly
clean studio shots while test images included real-world backgrounds. This likely
contributed to Model 1's poor generalization.



### Windows multiprocessing and memory

Running DataLoader with `NUM_WORKERS > 4` on Windows caused `WinError 1455`
(paging file too small) because each worker spawns a full PyTorch process.
Reducing `NUM_WORKERS` to 4 resolved the issue with no meaningful impact on
training speed given the dataset size.

## Model 1

Training accuracy in Model 1 exceeded 98% within the first few epochs while
validation accuracy stagnated — a textbook case of overfitting on a small dataset.
Dropout (0.5), data augmentation, and early stopping were applied, but without
pretrained features the model still failed to learn meaningful representations.


![Overfitting](images/overfitting.png)


The confusion matrix revealed the real problem: the model had a strong bias toward
predicting chocopuni, missing roughly half of all fumo images. This pointed to both
class imbalance and a mismatch between the training and test distributions -
training images were shot on controlled backgrounds, while test images came from
varied real-world settings.







![picture](images/m1.png)

## Model 2

Compared to Model 1, the confusion matrix shows a dramatically more balanced
prediction pattern. The model correctly identifies the majority of both classes,
with only a small number of errors in each direction.

- **3 Fumo images** were incorrectly predicted as Chocopuni (recall 0.82)
- **2 Chocopuni images** were incorrectly predicted as Fumo (recall 0.89)
- The near-diagonal matrix confirms the model is no longer biased toward
  either class — errors are scattered rather than systematic

The remaining errors likely correspond to ambiguous photos: unusual angles,
heavy occlusion, or lighting conditions not well represented in training data.





![picture](images/cm2.png)


### Training Results — K-Fold Cross-Validation

The model was trained using 3-fold stratified cross-validation.
All three folds converged smoothly without early stopping,
completing all 20 epochs with steadily decreasing validation loss.

| Fold | Val Accuracy | Val Loss (best) |
|------|-------------|-----------------|
| Fold 1 | 91.5% | 0.2789 |
| Fold 2 | 95.7% | 0.2599 |
| Fold 3 | 92.5% | 0.2494 |
| **Mean** | **93.2%** | — |
| **Std dev** | **±1.8%** | — |

The low standard deviation (±1.8%) indicates stable performance across different
data splits — the model is not sensitive to which images end up in validation.
**Fold 2** produced the best checkpoint and was selected for final evaluation.

### Test Results

The best fold checkpoint was evaluated on the held-out test set,
which was never seen during training or fold selection:

| Metric | Value |
|--------|-------|
| **Test Accuracy** | 85.4% |
| **Test Loss** | 0.474 |

The gap between mean validation accuracy (93.2%) and test accuracy (85.4%)
is notable — roughly 8 percentage points. This is expected given that the
test set contains images scraped from the internet with more varied backgrounds
and angles than the studio-style training photos, making it a harder and more
realistic evaluation.

![picture](images/m2.png)


## Model 3

### Training Results — K-Fold Cross-Validation

The model was trained using 5-fold stratified cross-validation with up to 40 epochs
per fold. Unlike Model 2, no early stopping triggered — all folds ran the full 40
epochs, reflecting the slower convergence expected when fine-tuning pretrained layers
with a very low learning rate (1e-6).

| Fold | Val Accuracy | Val Loss (best) |
|------|-------------|-----------------|
| Fold 1 | 100.0% | 0.1532 |
| Fold 2 | 92.9% | 0.2734 |
| Fold 3 | 92.9% | 0.2704 |
| Fold 4 | 94.6% | 0.2584 |
| Fold 5 | 91.1% | 0.2691 |
| **Mean** | **94.3%** | — |
| **Std dev** | **±3.1%** | — |

**Fold 1** achieved 100% validation accuracy and was selected as the best checkpoint.
This result should be interpreted with caution — the validation fold of 57 images is
small enough that a perfect score may reflect a particularly easy split rather than
true generalization. The higher standard deviation (±3.1%) compared to Model 2 (±1.8%)
confirms greater sensitivity to which images end up in each fold, which is expected
when more layers are being fine-tuned.

### Test Results

The best fold checkpoint (Fold 1) was evaluated on the held-out test set:

| Metric | Value |
|--------|-------|
| **Test Accuracy** | 91.7% |
| **Test Loss** | 0.392 |

The gap between mean validation accuracy (94.3%) and test accuracy (91.7%) is
approximately 2.6 percentage points — notably smaller than the 7.8 pp gap in
Model 2. This suggests that fine-tuning produced representations that generalize
better to real-world images from the internet, not just held-out studio photos.




![picture](images/m3.png)

### Per-Class Results

| Class | Precision | Recall | F1 | Support |
|-------|-----------|--------|----|---------|
| Chocopuni | 0.96 | 0.88 | 0.92 | 26 |
| Fumo | 0.88 | 0.95 | 0.91 | 22 |
| **Accuracy** | | | **0.92** | **48** |
| Macro avg | 0.92 | 0.92 | 0.92 | 48 |

Both classes are nearly perfectly balanced. Chocopuni precision (0.96) is the
highest single metric across all three models — when the model predicts Chocopuni,
it is almost always correct.

![picture](images/cm3.png)


## Reflection

These results show that transfer learning works best for small image datasets.
Model 1 (trained from scratch) quickly overfits and cannot reliably distinguish between Fumo and Chocopuni with only ~300 images.

This task is difficult because it is a fine-grained classification problem.
Both classes look very similar — same characters, colors, and style.
The differences are small (shape of the head, face, proportions), so the model needs to understand shapes, not just colors or textures.

That’s why pretrained models help so much.
Models trained on ImageNet already know how to detect shapes and objects, so they perform better.

Model 2 and Model 3 both use pretrained features.
The difference is that Model 3 allows more layers to adapt.
The improvement (+6.3%) suggests that the model learned details specific to plush toys.

In practice, the best approach is:

- start with a pretrained model (Model 2)
- then fine-tune it (Model 3) if needed

To improve results further, the most important thing is more and better data, not a more complex model.

![picture](images/mc.png)